# HLB-CIFAR10 — Ablation Study

Projet IDL

---
## Setup

In [ ]:
!git clone https://github.com/kckmzx/Project-hlb-CIFAR10
%cd Project-hlb-CIFAR10

In [ ]:
# Note: The one change we need to make if we're in Colab is to uncomment this below block.
# If we are in an ipython session or a notebook, clear the state to avoid bug
try:
  _ = get_ipython().__class__.__name__
  ## we set -f below to avoid prompting the user before clearing the notebook state
  %reset -f
except NameError:
  pass ## we're still good

In [ ]:
!python main.py --help

---
## 1. Baseline

**Objectif** : Établir la performance de référence avec toutes les techniques activées.

Tous les deltas de précision seront calculés par rapport à cette valeur. On utilisera le `mean_acc` de ce run comme 100% de référence.

L'idée est de reproduire la méthode décrite dans la Partie 1 par David Page, qui reproduit toujours un baseline propre avant de toucher quoi que ce soit.

In [ ]:
!python main.py --runs 5

---
## 2. Squeeze-and-Excite (SE)

**Objectif** : Mesurer la contribution des couches SE à la précision finale.

In [ ]:
!python main.py --runs 5 --no-se

---
## 3. Whitening

**Objectif** : Mesurer l'apport de la convolution de whitening en entrée du réseau.

In [ ]:
!python main.py --runs 5 --no-whitening

---
## 4. EMA — Exponential Moving Average

**Objectif** : Mesurer la contribution de l'EMA en fin d'entraînement.

**Raison** : En fin d'entraînement le LR est encore élevé — les poids oscillent autour du minimum. L'EMA moyenne les poids sur les derniers steps pour se placer au centre de ces oscillations, donc plus près du vrai minimum.

On veut comparer `ema_val_acc` vs `val_acc` dans la baseline pour quantifier l'apport de l'EMA seul. Tester aussi différentes durées pour trouver l'optimum.

In [ ]:
# EMA désactivé
!python main.py --runs 5 --no-ema

In [ ]:
# EMA sur 1 epoch
!python main.py --runs 5 --ema-epochs 1

In [ ]:
# EMA sur 4 epochs
!python main.py --runs 5 --ema-epochs 4

---
## 5. Test-Time Augmentation (TTA)

**Objectif** : Mesurer le gain apporté par le flip horizontal à l'évaluation.

À l'éval, le réseau (SpeedyResNet) voit chaque image et son miroir horizontal (`torch.flip`), puis moyenne les deux prédictions (`orig, flipped = x.split(x.shape[0]//2, dim=0)`).

In [ ]:
!python main.py --runs 5 --no-tta

---
## 6. Flip horizontal aléatoire

**Objectif** : Mesurer la contribution de l'augmentation par flip (`batch_flip_lr`)

In [ ]:
!python main.py --runs 5 --no-flip

---
## 7. Random Crop

**Objectif** : Mesurer la contribution du crop aléatoire à la généralisation.

On veut voir si c'est le crop ou le padding qui apporte le gain.


In [ ]:
# Sans crop, mais avec padding
!python main.py --runs 5 --no-crop

In [ ]:
# Sans crop ni padding
!python main.py --runs 5 --no-crop --pad-amount 0

---
## 8. Label Smoothing

**Objectif** : Mesurer l'effet de la régularisation par lissage des labels.

Le label smoothing est une technique de lissage moins définitive : au lieu de 100% de probabilité sur la classe correcte, on remplace la cible par "80% + 20% distribué sur les autres classes". Ça empêche le réseau d'être trop confiant.

In [ ]:
# Label smoothing désactivé
!python main.py --runs 5 --label-smoothing 0.0

In [ ]:
# Label smoothing 0.1
!python main.py --runs 5 --label-smoothing 0.1

In [ ]:
# Label smoothing 0.2 = baseline, déjà fait

In [ ]:
# Label smoothing 0.3
!python main.py --runs 5 --label-smoothing 0.3

---
## 9. Channels Last (optimisation mémoire)

TODO: REVOIR CETTE PARTIE: y'a des explications dans le code

**Objectif** : Mesurer le gain de vitesse du format mémoire NHWC sur les Tensor Cores, sans impact sur la précision.

**Raison** : PyTorch stocke par défaut en NCHW. Le format NHWC (channels_last) est nativement plus efficace pour les Tensor Cores Nvidia — les données sont contiguës dans le sens de lecture des convolutions.

**Comment lire les résultats** : Ici on regarde `total_time_seconds`, **pas** `val_acc` (la précision ne doit pas changer). Calculer le ratio temps avec/sans pour obtenir le speedup réel.

**Blog** : Post 1 — David Page montre que le GPU n'est utilisé qu'à ~12% de sa capacité théorique. Cette expérience quantifie un des facteurs responsables.

In [ ]:
!python main.py --runs 5 --no-channels-last

---
## 10. Learning Rate Schedule (warmup)

TODO: Rien compris

**Objectif** : Mesurer la sensibilité du modèle à la durée du warmup du OneCycleLR.

**Raison** : Le OneCycleLR monte le LR de ~0 jusqu'au maximum sur `percent_start × total_steps` steps. Trop court → instabilité en début d'entraînement. Trop long → temps gaspillé à un LR sous-optimal.

**Comment lire les résultats** : Observer `val_acc` à l'**epoch 0** selon `percent_start`. Sans warmup (0.0), l'epoch 0 devrait être catastrophique. L'epoch 0 se stabilise au-delà d'un certain seuil = durée minimale de warmup utile.

**Blog** : Post 2 — le warmup est la solution nécessaire quand on monte le LR avec le batch size. Post 5 — la direction plate λ/(1−ρ) explique pourquoi modifier le schedule ne casse pas tout.

In [ ]:
# Pas de warmup
!python main.py --runs 5 --percent-start 0.0

In [ ]:
# Warmup 10%
!python main.py --runs 5 --percent-start 0.1

In [ ]:
# Warmup 20% = baseline, déjà fait

In [ ]:
# Warmup 35%
!python main.py --runs 5 --percent-start 0.35

In [ ]:
# Warmup 50%
!python main.py --runs 5 --percent-start 0.5

---
## 11. Règle de mise à l'échelle linéaire (batch size)

**Objectif** : Vérifier que multiplier le batch size par n et le LR par n conserve la précision finale.

**Raison** : C'est la thèse centrale du post 2. Un grand step (LR×n) avec un gradient plus précis (batch×n) est équivalent à n petits steps.

**Comment lire les résultats** :
- Trois `mean_acc` statistiquement identiques → la règle tient ✓
- batch=1024 dégrade → limite du régime linéaire trouvée
- `total_time_seconds` : batch=1024 devrait être ~2× plus rapide que batch=256

**Blog** : Post 2 — thèse centrale. Post 5 — la direction plate λ/N = constante est la formalisation théorique.

In [ ]:
# Batch 256, LR /2
!python main.py --runs 5 --batchsize 256 --non-bias-lr-scale 0.5 --bias-lr-scale 0.5

In [ ]:
# Batch 512 = baseline, déjà fait

In [ ]:
# Batch 1024, LR x2
!python main.py --runs 5 --batchsize 1024 --non-bias-lr-scale 2.0 --bias-lr-scale 2.0

In [ ]:
# Batch 1024, LR x2, warmup plus long (si le précédent dégrade)
!python main.py --runs 5 --batchsize 1024 --non-bias-lr-scale 2.0 --bias-lr-scale 2.0 --percent-start 0.35

---
## 12. Référence basse — tout désactivé

**Objectif** : Mesurer la performance sans aucune technique.

L'écart entre cette valeur et la baseline représente la contribution **totale** de toutes les techniques réunies.

In [ ]:
!python main.py --runs 5 --no-se --no-whitening --no-ema --no-tta --no-flip --no-crop --pad-amount 0 --label-smoothing 0.0 --no-channels-last